# Local Photo ONNX Fixture Test

This notebook runs the exported ONNX model from `hallway_lighting_runs/exports/` on an uploaded hallway image.

It now does three things that matter for reliability:
- uses the same ImageNet normalization that the training pipeline used
- rejects near-black frames instead of trusting obviously invalid positive lux output
- estimates fixture count, under-fixture lux, and between-adjacent-fixture lux with an overlay

For best results, use an end-to-end hallway photo where the floor is visible in the lower part of the frame and the ceiling fixtures appear one after another down the corridor.


In [ ]:
%pip install onnxruntime pillow matplotlib ipywidgets numpy

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Arwin-K/APS112-DL-Lighting-Camera.git"
COLAB_REPO_DIR = Path("/content/APS112-DL-Lighting-Camera")

if not COLAB_REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
else:
    print(f"Using existing runtime clone: {COLAB_REPO_DIR}")


In [ ]:
from __future__ import annotations

import importlib.util
from pathlib import Path
import sys
from typing import Any

import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display
import matplotlib.pyplot as plt


PROJECT_ROOT_OVERRIDE: Path | None = None
MODEL_FILENAME = "hallway_multitask_unet_drive_prototype.onnx"
COLAB_PROJECT_ROOT = Path("/content/APS112-DL-Lighting-Camera/hallway_lighting")


def discover_notebook_project_root() -> Path:
    candidates: list[Path] = []
    if PROJECT_ROOT_OVERRIDE is not None:
        candidates.append(PROJECT_ROOT_OVERRIDE.expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([
        COLAB_PROJECT_ROOT,
        cwd,
        cwd / "hallway_lighting",
    ])

    current = cwd
    for _ in range(6):
        candidates.append(current)
        candidates.append(current / "hallway_lighting")
        if current.parent == current:
            break
        current = current.parent

    seen: set[Path] = set()
    ordered_candidates: list[Path] = []
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            continue
        if resolved not in seen:
            seen.add(resolved)
            ordered_candidates.append(resolved)

    for candidate in ordered_candidates:
        model_path = candidate / "hallway_lighting_runs" / "exports" / MODEL_FILENAME
        fixture_path = candidate / "hallway_lighting" / "utils" / "fixture_detection.py"
        if model_path.exists() and fixture_path.exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the hallway_lighting project folder and exported ONNX model. "
        "If you are in Colab, run the clone cell first. Otherwise open the notebook from inside the repo. "
        f"Candidates checked: {[str(path) for path in ordered_candidates]}"
    )


PROJECT_ROOT = discover_notebook_project_root()
DEFAULT_ONNX_PATH = PROJECT_ROOT / "hallway_lighting_runs" / "exports" / MODEL_FILENAME
OUTPUT_DIR = PROJECT_ROOT / "hallway_lighting_runs" / "local_photo_tests"
FIXTURE_DETECTION_PATH = PROJECT_ROOT / "hallway_lighting" / "utils" / "fixture_detection.py"


def load_fixture_detection_module():
    spec = importlib.util.spec_from_file_location("fixture_detection_local", FIXTURE_DETECTION_PATH)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load fixture detection helper from {FIXTURE_DETECTION_PATH}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module


fixture_detection = load_fixture_detection_module()
infer_fixture_layout = fixture_detection.infer_fixture_layout

HELPER_SOURCE = '"""Helpers for running exported ONNX inference from the local notebook."""\n\nfrom __future__ import annotations\n\nfrom io import BytesIO\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import Polygon\nimport numpy as np\nfrom PIL import Image\n\n# infer_fixture_layout is injected by the notebook before exec()\n\nMODEL_FILENAME = "hallway_multitask_unet_drive_prototype.onnx"\nCOLAB_PROJECT_ROOT = Path("/content/APS112-DL-Lighting-Camera/hallway_lighting")\nIMAGENET_MEAN = np.asarray([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 3)\nIMAGENET_STD = np.asarray([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 3)\nDARK_FRAME_MEAN_THRESHOLD = 0.03\nDARK_FRAME_P95_THRESHOLD = 0.08\n\n\ndef discover_project_root(project_root_override: Path | None = None) -> Path:\n    """Locates the outer hallway_lighting project directory."""\n\n    candidates: list[Path] = []\n    if project_root_override is not None:\n        candidates.append(project_root_override.expanduser())\n\n    cwd = Path.cwd().resolve()\n    candidates.extend([COLAB_PROJECT_ROOT, cwd, cwd / "hallway_lighting"])\n\n    current = cwd\n    for _ in range(6):\n        candidates.append(current)\n        candidates.append(current / "hallway_lighting")\n        if current.parent == current:\n            break\n        current = current.parent\n\n    seen: set[Path] = set()\n    ordered_candidates: list[Path] = []\n    for candidate in candidates:\n        try:\n            resolved = candidate.resolve()\n        except Exception:\n            continue\n        if resolved not in seen:\n            seen.add(resolved)\n            ordered_candidates.append(resolved)\n\n    for candidate in ordered_candidates:\n        model_path = candidate / "hallway_lighting_runs" / "exports" / MODEL_FILENAME\n        fixture_path = candidate / "hallway_lighting" / "utils" / "fixture_detection.py"\n        if model_path.exists() and fixture_path.exists():\n            return candidate\n\n    raise FileNotFoundError(\n        "Could not locate the hallway_lighting project folder and exported ONNX model. "\n        "If you are in Colab, run the clone cell first. Otherwise open the notebook from inside the repo. "\n        f"Candidates checked: {[str(path) for path in ordered_candidates]}"\n    )\n\n\ndef default_onnx_path(project_root: Path) -> Path:\n    return project_root / "hallway_lighting_runs" / "exports" / MODEL_FILENAME\n\n\ndef default_output_dir(project_root: Path) -> Path:\n    return project_root / "hallway_lighting_runs" / "local_photo_tests"\n\n\ndef load_onnx_session(onnx_path: Path):\n    """Loads the ONNX runtime session on CPU."""\n\n    try:\n        import onnxruntime as ort\n    except ImportError as exc:\n        raise ImportError(\n            "onnxruntime is not installed in this notebook environment. "\n            "Run the install cell, restart the kernel, and rerun the notebook."\n        ) from exc\n\n    if not onnx_path.exists():\n        raise FileNotFoundError(f"ONNX model not found: {onnx_path}")\n    return ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])\n\n\ndef extract_single_map(value: np.ndarray | float | int | None) -> np.ndarray | None:\n    """Returns a single 2D or HxWxC array from an ONNX output value."""\n\n    if value is None or isinstance(value, (float, int)):\n        return None\n    array = np.asarray(value)\n    if array.ndim == 4:\n        array = array[0]\n    if array.ndim == 3 and array.shape[0] in {1, 3}:\n        array = np.transpose(array, (1, 2, 0))\n    if array.ndim == 3 and array.shape[-1] == 1:\n        array = array[..., 0]\n    return array.astype(np.float32, copy=False)\n\n\ndef extract_scalar(value: np.ndarray | float | int | None, fallback: float = 0.0) -> float:\n    """Extracts a scalar from a tensor-like output."""\n\n    if value is None:\n        return fallback\n    if isinstance(value, (float, int)):\n        return float(value)\n    array = np.asarray(value).reshape(-1)\n    return fallback if array.size == 0 else float(array[0])\n\n\ndef assess_image_quality(display_rgb: np.ndarray) -> dict[str, float | bool]:\n    """Computes simple brightness metrics and a dark-frame gate."""\n\n    rgb = np.clip(np.asarray(display_rgb, dtype=np.float32), 0.0, 1.0)\n    luminance = 0.2126 * rgb[..., 0] + 0.7152 * rgb[..., 1] + 0.0722 * rgb[..., 2]\n    bottom_start = min(rgb.shape[0] - 1, max(0, int(round(rgb.shape[0] * 0.55))))\n    bottom_band = luminance[bottom_start:, :] if bottom_start < rgb.shape[0] else luminance\n\n    mean_luminance = float(np.mean(luminance))\n    p95_luminance = float(np.percentile(luminance, 95))\n    max_luminance = float(np.max(luminance))\n    bottom_mean_luminance = float(np.mean(bottom_band))\n    is_dark_frame = (\n        mean_luminance < DARK_FRAME_MEAN_THRESHOLD\n        and p95_luminance < DARK_FRAME_P95_THRESHOLD\n    )\n    return {\n        "mean_luminance": mean_luminance,\n        "p95_luminance": p95_luminance,\n        "max_luminance": max_luminance,\n        "bottom_mean_luminance": bottom_mean_luminance,\n        "is_dark_frame": bool(is_dark_frame),\n    }\n\n\ndef preprocess_uploaded_image(\n    image_bytes: bytes,\n    input_height: int,\n    input_width: int,\n) -> tuple[np.ndarray, np.ndarray, dict[str, float | bool]]:\n    """Resizes image, keeps a display copy, and applies training-time normalization."""\n\n    image = Image.open(BytesIO(image_bytes)).convert("RGB")\n    resized = image.resize((input_width, input_height), resample=Image.BILINEAR)\n    display_rgb = np.asarray(resized).astype(np.float32) / 255.0\n    quality = assess_image_quality(display_rgb)\n    normalized_rgb = (display_rgb - IMAGENET_MEAN) / IMAGENET_STD\n    model_input = np.transpose(normalized_rgb, (2, 0, 1))[None, ...].astype(np.float32)\n    return model_input, display_rgb, quality\n\n\ndef sample_bilinear(map_2d: np.ndarray, x_normalized: float, y_normalized: float) -> float:\n    """Samples a 2D map in normalized coordinates."""\n\n    height, width = map_2d.shape\n    x = float(np.clip(x_normalized, 0.0, 1.0)) * (width - 1)\n    y = float(np.clip(y_normalized, 0.0, 1.0)) * (height - 1)\n    x0 = int(np.floor(x))\n    x1 = min(x0 + 1, width - 1)\n    y0 = int(np.floor(y))\n    y1 = min(y0 + 1, height - 1)\n    wx = x - x0\n    wy = y - y0\n    top = (1.0 - wx) * float(map_2d[y0, x0]) + wx * float(map_2d[y0, x1])\n    bottom = (1.0 - wx) * float(map_2d[y1, x0]) + wx * float(map_2d[y1, x1])\n    return (1.0 - wy) * top + wy * bottom\n\n\ndef sample_point_lux(lux_map: np.ndarray, point_targets: list[Any]) -> dict[str, float]:\n    """Samples lux values for the projected under/between-fixture targets."""\n\n    return {point.name: sample_bilinear(lux_map, point.x, point.y) for point in point_targets}\n\n\ndef summarize_lux_map(lux_map: np.ndarray, floor_mask: np.ndarray | None = None) -> dict[str, float]:\n    """Computes basic lux summary statistics."""\n\n    values = np.asarray(lux_map, dtype=np.float32)\n    if floor_mask is not None:\n        mask = np.asarray(floor_mask).astype(bool)\n        if mask.shape == values.shape and float(mask.mean()) > 0.001:\n            values = values[mask]\n        else:\n            values = values.reshape(-1)\n    else:\n        values = values.reshape(-1)\n\n    if values.size == 0:\n        return {"avg_lux": 0.0, "low_lux_p5": 0.0, "high_lux_p95": 0.0}\n    return {\n        "avg_lux": float(np.mean(values)),\n        "low_lux_p5": float(np.percentile(values, 5)),\n        "high_lux_p95": float(np.percentile(values, 95)),\n    }\n\n\ndef build_overlay_figure(\n    display_rgb: np.ndarray,\n    lux_map: np.ndarray,\n    fixture_analysis: dict[str, Any] | None,\n    point_lux: dict[str, float],\n    warning_texts: list[str] | None = None,\n):\n    """Builds the inline/saveable notebook overlay."""\n\n    figure, axis = plt.subplots(1, 1, figsize=(8, 5))\n    axis.imshow(display_rgb)\n    if float(np.max(lux_map, initial=0.0)) > 1e-6:\n        axis.imshow(lux_map, cmap="inferno", alpha=0.38)\n\n    height, width = display_rgb.shape[:2]\n    warning_texts = warning_texts or []\n\n    if fixture_analysis is not None:\n        for region in fixture_analysis.get("between_regions", []):\n            polygon_points = region.get("polygon") or []\n            if polygon_points:\n                polygon_pixels = [\n                    (float(x_value) * (width - 1), float(y_value) * (height - 1))\n                    for x_value, y_value in polygon_points\n                ]\n                axis.add_patch(\n                    Polygon(\n                        polygon_pixels,\n                        closed=True,\n                        facecolor="#80cbc4",\n                        edgecolor="#004d40",\n                        linewidth=1.2,\n                        alpha=0.18,\n                    )\n                )\n\n        for fixture in fixture_analysis.get("fixtures", []):\n            x_pixel = float(fixture["x"]) * (width - 1)\n            y_pixel = float(fixture["y"]) * (height - 1)\n            axis.scatter(\n                x_pixel,\n                y_pixel,\n                s=120,\n                c="#ffee58",\n                edgecolors="black",\n                linewidths=0.9,\n                zorder=3,\n            )\n            axis.text(\n                x_pixel + 6,\n                y_pixel - 6,\n                str(fixture["name"]),\n                color="white",\n                fontsize=8,\n                bbox={"facecolor": "black", "alpha": 0.45, "pad": 2},\n            )\n\n        for point in fixture_analysis.get("point_targets", []):\n            point_name = str(point["name"])\n            if point_name not in point_lux:\n                continue\n            x_pixel = float(point["x"]) * (width - 1)\n            y_pixel = float(point["y"]) * (height - 1)\n            point_color = "#4fc3f7" if point_name.startswith("between_") else "#ffffff"\n            axis.scatter(\n                x_pixel,\n                y_pixel,\n                s=48,\n                c=point_color,\n                edgecolors="black",\n                linewidths=0.8,\n                zorder=4,\n            )\n            axis.text(\n                x_pixel + 4,\n                y_pixel + 10,\n                f"{point_name}\\n{point_lux[point_name]:.1f} lux",\n                color="white",\n                fontsize=7,\n                bbox={"facecolor": "black", "alpha": 0.5, "pad": 2},\n                zorder=5,\n            )\n\n    if warning_texts:\n        axis.text(\n            0.01,\n            0.99,\n            "\\n".join(warning_texts),\n            transform=axis.transAxes,\n            ha="left",\n            va="top",\n            color="white",\n            fontsize=8,\n            bbox={"facecolor": "#5d4037", "alpha": 0.72, "pad": 4},\n            zorder=6,\n        )\n\n    axis.set_title("Fixture-aware Lux Overlay")\n    axis.axis("off")\n    figure.tight_layout()\n    return figure\n\n\ndef save_result_artifacts(\n    output_dir: Path,\n    image_name: str,\n    display_rgb: np.ndarray,\n    lux_map: np.ndarray,\n    fixture_analysis: dict[str, Any] | None,\n    point_lux: dict[str, float],\n    result: dict[str, Any],\n    warning_texts: list[str],\n):\n    """Saves the overlay and JSON summary for a notebook run."""\n\n    figure = build_overlay_figure(\n        display_rgb=display_rgb,\n        lux_map=lux_map,\n        fixture_analysis=fixture_analysis,\n        point_lux=point_lux,\n        warning_texts=warning_texts,\n    )\n    output_dir.mkdir(parents=True, exist_ok=True)\n    image_stem = Path(image_name).stem\n    overlay_path = output_dir / f"{image_stem}_fixture_lux_overlay.png"\n    summary_path = output_dir / f"{image_stem}_fixture_lux_summary.json"\n    figure.savefig(overlay_path, bbox_inches="tight")\n    result["overlay_image"] = str(overlay_path)\n    result["summary_json"] = str(summary_path)\n    with summary_path.open("w", encoding="utf-8") as handle:\n        json.dump(result, handle, indent=2)\n    return figure\n\n\ndef run_uploaded_photo(\n    image_bytes: bytes,\n    image_name: str,\n    onnx_path: Path,\n    output_dir: Path,\n    max_fixture_count: int = 8,\n    floor_area_m2: float = 12.0,\n):\n    """Runs ONNX inference plus fixture-aware post-processing for the notebook."""\n\n    session = load_onnx_session(onnx_path)\n    input_shape = session.get_inputs()[0].shape\n    input_height = int(input_shape[2])\n    input_width = int(input_shape[3])\n    model_input, display_rgb, image_quality = preprocess_uploaded_image(\n        image_bytes=image_bytes,\n        input_height=input_height,\n        input_width=input_width,\n    )\n\n    warning_texts: list[str] = []\n    if bool(image_quality["is_dark_frame"]):\n        warning_texts.append(\n            "Frame is too dark for reliable lux inference. Returning 0 lux and no fixtures."\n        )\n        result = {\n            "image_name": image_name,\n            "onnx_path": str(onnx_path),\n            "fixture_count": 0,\n            "avg_lux": 0.0,\n            "low_lux_p5": 0.0,\n            "high_lux_p95": 0.0,\n            "lux_map_summary": {"avg_lux": 0.0, "low_lux_p5": 0.0, "high_lux_p95": 0.0},\n            "under_fixture_lux": {},\n            "between_fixture_lux": {},\n            "point_lux": {},\n            "fixture_analysis": None,\n            "warnings": warning_texts,\n            "image_quality": image_quality,\n            "inference_skipped": True,\n        }\n        zero_lux_map = np.zeros((input_height, input_width), dtype=np.float32)\n        figure = save_result_artifacts(\n            output_dir=output_dir,\n            image_name=image_name,\n            display_rgb=display_rgb,\n            lux_map=zero_lux_map,\n            fixture_analysis=None,\n            point_lux={},\n            result=result,\n            warning_texts=warning_texts,\n        )\n        return result, figure\n\n    raw_outputs = session.run(None, {session.get_inputs()[0].name: model_input})\n    output_names = [\n        "lux_map",\n        "avg_lux",\n        "low_lux_p5",\n        "high_lux_p95",\n        "floor_mask_pred",\n        "albedo_pred",\n        "gloss_pred",\n        "uncertainty_map",\n        "estimated_power_w",\n    ]\n    named_outputs = {name: value for name, value in zip(output_names, raw_outputs)}\n\n    lux_map = extract_single_map(named_outputs["lux_map"])\n    if lux_map is None or lux_map.ndim != 2:\n        raise ValueError("Expected a single-channel lux map from the ONNX output.")\n\n    floor_mask = extract_single_map(named_outputs.get("floor_mask_pred"))\n    floor_mask_binary = None if floor_mask is None else (np.asarray(floor_mask) > 0.5)\n\n    floor_mask_ratio = None\n    if floor_mask_binary is not None:\n        floor_mask_ratio = float(np.mean(floor_mask_binary))\n        if floor_mask_ratio < 0.03:\n            warning_texts.append(\n                "Predicted floor area is very small in this frame, so fixture placement may be unreliable."\n            )\n\n    fixture_layout = infer_fixture_layout(\n        image=display_rgb,\n        floor_mask=floor_mask_binary,\n        max_fixture_count=max_fixture_count,\n        floor_area_m2=floor_area_m2,\n    )\n    fixture_analysis = None if fixture_layout is None else fixture_layout.to_summary_dict()\n    point_targets = [] if fixture_layout is None else list(fixture_layout.point_targets)\n    point_lux = sample_point_lux(lux_map, point_targets)\n    under_fixture_lux = {name: value for name, value in point_lux.items() if name.startswith("under_")}\n    between_fixture_lux = {\n        name: value for name, value in point_lux.items() if name.startswith("between_")\n    }\n\n    if fixture_analysis is None or int(fixture_analysis.get("inferred_fixture_count", 0)) == 0:\n        warning_texts.append(\n            "No fixtures were found in this frame. Use an end-to-end hallway image with visible floor "\n            "and increase Max fixtures if needed."\n        )\n    elif int(fixture_analysis.get("inferred_fixture_count", 0)) >= int(max_fixture_count):\n        warning_texts.append(\n            "Detected fixture count reached the Max fixtures cap. Increase that value if your hallway has more fixtures."\n        )\n\n    lux_summary = summarize_lux_map(lux_map=lux_map, floor_mask=floor_mask_binary)\n    result = {\n        "image_name": image_name,\n        "onnx_path": str(onnx_path),\n        "fixture_count": 0 if fixture_analysis is None else int(fixture_analysis.get("inferred_fixture_count", 0)),\n        "avg_lux": extract_scalar(named_outputs.get("avg_lux"), fallback=lux_summary["avg_lux"]),\n        "low_lux_p5": extract_scalar(named_outputs.get("low_lux_p5"), fallback=lux_summary["low_lux_p5"]),\n        "high_lux_p95": extract_scalar(\n            named_outputs.get("high_lux_p95"),\n            fallback=lux_summary["high_lux_p95"],\n        ),\n        "lux_map_summary": lux_summary,\n        "under_fixture_lux": under_fixture_lux,\n        "between_fixture_lux": between_fixture_lux,\n        "point_lux": point_lux,\n        "fixture_analysis": fixture_analysis,\n        "warnings": warning_texts,\n        "image_quality": image_quality,\n        "inference_skipped": False,\n        "floor_mask_ratio": floor_mask_ratio,\n    }\n    figure = save_result_artifacts(\n        output_dir=output_dir,\n        image_name=image_name,\n        display_rgb=display_rgb,\n        lux_map=lux_map,\n        fixture_analysis=fixture_analysis,\n        point_lux=point_lux,\n        result=result,\n        warning_texts=warning_texts,\n    )\n    return result, figure\n\n'
exec(HELPER_SOURCE, globals())

print(f"Project root: {PROJECT_ROOT}")
print(f"Default ONNX: {DEFAULT_ONNX_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print("Notebook preprocessing: ImageNet mean/std normalization enabled")
print(
    f"Dark-frame gate: mean<{DARK_FRAME_MEAN_THRESHOLD:.2f} and p95<{DARK_FRAME_P95_THRESHOLD:.2f} -> skip inference"
)


In [ ]:
upload_widget = widgets.FileUpload(accept="image/*", multiple=False, description="Upload photo")
max_fixtures_widget = widgets.BoundedIntText(value=12, min=1, max=30, description="Max fixtures")
floor_area_widget = widgets.FloatText(value=12.0, description="Floor area m2")
run_button = widgets.Button(description="Run ONNX test", button_style="primary")
output_widget = widgets.Output()


def extract_uploaded_entry(upload_value):
    if isinstance(upload_value, dict) and upload_value:
        first_key = next(iter(upload_value.keys()))
        first_value = upload_value[first_key]
        if isinstance(first_value, dict):
            image_name = first_value.get("name", first_key)
            image_bytes = first_value.get("content")
            return image_name, image_bytes
        return str(first_key), first_value

    if isinstance(upload_value, (list, tuple)) and upload_value:
        first_value = upload_value[0]
        if isinstance(first_value, dict):
            image_name = first_value.get("name") or first_value.get("metadata", {}).get("name") or "uploaded_image"
            image_bytes = first_value.get("content")
            return image_name, image_bytes
        return "uploaded_image", first_value

    return None, None


def render_result(result: dict[str, Any]):
    display(Markdown(f"**Detected fixtures:** {result['fixture_count']}"))
    display(Markdown(f"**Average lux:** {result['avg_lux']:.1f}"))

    quality = result.get("image_quality") or {}
    if quality:
        print(
            "Image brightness: "
            f"mean={float(quality.get('mean_luminance', 0.0)):.3f}, "
            f"p95={float(quality.get('p95_luminance', 0.0)):.3f}, "
            f"bottom_mean={float(quality.get('bottom_mean_luminance', 0.0)):.3f}"
        )

    warnings = result.get("warnings") or []
    if warnings:
        print("Warnings:")
        for warning in warnings:
            print(f"  - {warning}")

    print("Under-fixture lux:")
    if result["under_fixture_lux"]:
        for point_name, lux_value in result["under_fixture_lux"].items():
            print(f"  - {point_name}: {lux_value:.1f} lux")
    else:
        print("  - none detected")

    print("Between adjacent fixtures lux:")
    if result["between_fixture_lux"]:
        for point_name, lux_value in result["between_fixture_lux"].items():
            print(f"  - {point_name}: {lux_value:.1f} lux")
    else:
        print("  - none detected")

    print(f"Overlay saved to: {result['overlay_image']}")
    print(f"Summary saved to: {result['summary_json']}")


def on_run_clicked(_):
    with output_widget:
        clear_output()
        image_name, image_bytes = extract_uploaded_entry(upload_widget.value)
        if image_bytes is None:
            print("Upload one image first.")
            return

        try:
            result, figure = run_uploaded_photo(
                image_bytes=image_bytes,
                image_name=image_name,
                onnx_path=DEFAULT_ONNX_PATH,
                output_dir=OUTPUT_DIR,
                max_fixture_count=int(max_fixtures_widget.value),
                floor_area_m2=float(floor_area_widget.value),
            )
        except Exception as exc:
            print(f"Inference failed: {exc}")
            return

        render_result(result)
        display(figure)
        plt.close(figure)


run_button.on_click(on_run_clicked)

display(
    widgets.VBox(
        [
            widgets.HTML("<b>Upload a hallway photo and run exported ONNX inference</b>"),
            widgets.HTML(f"<code>{DEFAULT_ONNX_PATH}</code>"),
            widgets.HTML(
                "Uses training-time ImageNet normalization and rejects near-black frames. "
                "Best results come from end-to-end hallway images with visible floor."
            ),
            upload_widget,
            widgets.HBox([max_fixtures_widget, floor_area_widget]),
            run_button,
            output_widget,
        ]
    )
)
